# 03 — Crypto spot-perp funding arb backtest

**Setup:** long spot, short perp (1:1 notional). Collect funding. Unwind on either:
- N consecutive negative funding prints, or
- |basis| > threshold (suggests stress).

**Data:** Binance public REST for funding history (no auth). yfinance for spot proxy.

**Mode:** backtest only. Paper trading lives in NB 07.

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from src.data.crypto_clients import BinanceClient
from src.data.yfinance_client import YFinanceClient
from src.strategies.crypto_funding_arb import simulate_funding_arb, FundingArbConfig

bnc = BinanceClient()
f = bnc.get_funding_history('BTCUSDT', start='2022-01-01')
print('Funding rows:', len(f))
f.head()

In [ ]:
yf = YFinanceClient()
btc = yf.get_daily_bars('BTC-USD', start='2022-01-01')
spot = btc['adj_close']
spot.index = spot.index.tz_localize(None)
print('BTC daily rows:', len(spot))

### Simulate

In [ ]:
res = simulate_funding_arb(
    funding=f['rate'],
    spot=spot,
    perp=None,                       # treat perp = spot in absence of perp marks
    cfg=FundingArbConfig(notional=100_000, unwind_funding_threshold=0.0,
                          unwind_periods=6, unwind_basis_bps=80))
res['metrics']

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(res['equity'].index, res['equity'].values, linewidth=1)
ax[0].set_title('Funding arb equity (notional $100k)')
ax[0].grid(alpha=0.3)
ax[1].plot(res['daily_funding'].index, res['daily_funding'].values * 10_000, linewidth=0.6)
ax[1].set_ylabel('daily funding (bp)')
ax[1].grid(alpha=0.3); plt.show()

print(f'unwind events: {len(res["events"])}')
for d, why in res['events'][:5]: print(f'  {d.date()}: {why}')

### Interpretation & risk

Real funding-arb risk in production includes:
- **Counterparty risk** — exchange insolvency wipes the leg you couldn't withdraw.
- **Collateral risk** — perp short uses USDT margin; USDT depeg = liquidation.
- **Slippage at unwind** — sized small (1-2bp on liquid pairs) but real.
- **Borrow / financing** — long spot doesn't borrow, but margin perp does.

See `RISK_POLICY.md` for the venue-allocation cap and stress-test triggers.